# nanoGPT Scaling Study (Colab)

**Before you start:** Runtime > Change runtime type > **GPU (T4)**. Then run the cells in order.

What you will do: (1) write the project files, (2) run sanity tests, (3) run one quick training, (4) run the scaling sweep, (5) plot and save results.

If Colab disconnects, re-run the first cells and then the sweep cell: finished runs are skipped.

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('No GPU: go to Runtime > Change runtime type and choose a GPU.')

## Optional: save results to Google Drive
Run this if you want results to survive a disconnect. Set `RESULTS_DIR` to a Drive folder.

In [ ]:
import os
USE_DRIVE = False   # set True to save results to Google Drive
RESULTS_DIR = 'results'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/nanogpt_results'
os.makedirs('tests', exist_ok=True)
print('Results will be saved to', RESULTS_DIR)

## 1. Project files
Each cell below writes one file. Read them: you need to be able to explain every line in an interview.

In [ ]:
%%writefile data.py
"""Character-level dataset (Tiny Shakespeare by default)."""
import os
import urllib.request

import torch

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"


def load_text(path="data/input.txt"):
    """Load the corpus, downloading Tiny Shakespeare on first use."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(URL, path)
        except Exception as e:  # network problems
            raise RuntimeError(
                f"Could not download the dataset ({e}). Download it manually from\n{URL}\n"
                f"and save it as {path}."
            )
    with open(path, encoding="utf-8") as f:
        return f.read()


class CharDataset:
    def __init__(self, text, val_frac=0.1):
        chars = sorted(set(text))
        self.chars = chars
        self.stoi = {c: i for i, c in enumerate(chars)}
        self.itos = {i: c for i, c in enumerate(chars)}
        self.vocab_size = len(chars)
        data = torch.tensor([self.stoi[c] for c in text], dtype=torch.long)
        n = int(len(data) * (1 - val_frac))
        self.train, self.val = data[:n], data[n:]

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)

    def get_batch(self, split, batch_size, block_size, device, generator=None):
        d = self.train if split == "train" else self.val
        ix = torch.randint(len(d) - block_size - 1, (batch_size,), generator=generator)
        x = torch.stack([d[i : i + block_size] for i in ix])
        y = torch.stack([d[i + 1 : i + 1 + block_size] for i in ix])
        return x.to(device), y.to(device)

In [ ]:
%%writefile model.py
"""A small GPT (decoder-only transformer) written from scratch in PyTorch."""
import math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    block_size: int = 128
    vocab_size: int = 65
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1
    ln_placement: str = "pre"  # "pre" (LayerNorm before sublayers) or "post" (original Transformer)

    def to_dict(self):
        return asdict(self)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.attn_dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # Each position may only attend to itself and earlier positions (is_causal=True).
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.attn_dropout if self.training else 0.0, is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.ln_placement in ("pre", "post")
        self.pre = cfg.ln_placement == "pre"
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        if self.pre:
            x = x + self.attn(self.ln1(x))
            x = x + self.mlp(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.mlp(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd) if cfg.ln_placement == "pre" else nn.Identity()
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight  # weight tying
        self.apply(self._init_weights)
        # scaled init for residual projections (GPT-2 style)
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())  # tied weights are counted once
        if non_embedding:
            n -= self.wte.weight.numel() + self.wpe.weight.numel()
        return n

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"sequence length {T} > block_size {self.cfg.block_size}"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.wte(idx) + self.wpe(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressively sample tokens. Call model.eval() first."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
%%writefile train.py
"""Training loop for the small GPT. Use as a library (train(**overrides)) or from the CLI."""
import argparse
import json
import math
import os
import time
from contextlib import nullcontext

import torch

from data import CharDataset, load_text
from model import GPT, GPTConfig

DEFAULTS = dict(
    block_size=128, batch_size=64,
    n_layer=4, n_head=4, n_embd=128, dropout=0.1, ln_placement="pre",
    max_iters=3000, eval_interval=250, eval_iters=50,
    lr=1e-3, min_lr=1e-4, warmup=100, weight_decay=0.1, grad_clip=1.0,
    seed=0, out_dir="out/run", save_ckpt=True,
)


def get_lr(it, max_iters, lr, min_lr, warmup):
    """Linear warmup, then cosine decay to min_lr."""
    if it < warmup:
        return lr * (it + 1) / warmup
    if it >= max_iters:
        return min_lr
    ratio = (it - warmup) / max(1, max_iters - warmup)
    return min_lr + 0.5 * (1 + math.cos(math.pi * ratio)) * (lr - min_lr)


@torch.no_grad()
def estimate_loss(model, ds, cfg, device, amp_ctx):
    """Average loss over fixed batches (same batches every call, so numbers are comparable)."""
    model.eval()
    out = {}
    for split in ("train", "val"):
        g = torch.Generator().manual_seed(1234)
        losses = []
        for _ in range(cfg["eval_iters"]):
            x, y = ds.get_batch(split, cfg["batch_size"], cfg["block_size"], device, g)
            with amp_ctx:
                _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out


def train(**overrides):
    cfg = {**DEFAULTS, **overrides}
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(cfg["seed"])

    ds = CharDataset(load_text())
    mc = GPTConfig(
        block_size=cfg["block_size"], vocab_size=ds.vocab_size, n_layer=cfg["n_layer"],
        n_head=cfg["n_head"], n_embd=cfg["n_embd"], dropout=cfg["dropout"],
        ln_placement=cfg["ln_placement"],
    )
    model = GPT(mc).to(device)
    n_params = model.num_params()
    n_params_ne = model.num_params(non_embedding=True)

    use_cuda = device == "cuda"
    amp_dtype = torch.bfloat16 if (use_cuda and torch.cuda.is_bf16_supported()) else torch.float16
    amp_ctx = torch.autocast(device_type="cuda", dtype=amp_dtype) if use_cuda else nullcontext()
    scaler = torch.amp.GradScaler("cuda", enabled=(use_cuda and amp_dtype == torch.float16))

    # no weight decay on biases and LayerNorm params (1-D tensors)
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg["weight_decay"]}, {"params": no_decay, "weight_decay": 0.0}],
        lr=cfg["lr"], betas=(0.9, 0.95),
    )

    g = torch.Generator().manual_seed(cfg["seed"])
    history, t0 = [], time.time()
    for it in range(cfg["max_iters"] + 1):
        lr = get_lr(it, cfg["max_iters"], cfg["lr"], cfg["min_lr"], cfg["warmup"])
        for group in optimizer.param_groups:
            group["lr"] = lr

        if it % cfg["eval_interval"] == 0 or it == cfg["max_iters"]:
            l = estimate_loss(model, ds, cfg, device, amp_ctx)
            history.append(dict(iter=it, train=l["train"], val=l["val"], lr=lr, seconds=time.time() - t0))
            print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f} | lr {lr:.2e} | {time.time()-t0:.0f}s", flush=True)
        if it == cfg["max_iters"]:
            break

        x, y = ds.get_batch("train", cfg["batch_size"], cfg["block_size"], device, g)
        with amp_ctx:
            _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        scaler.step(optimizer)
        scaler.update()

    result = dict(
        config=cfg, n_params=n_params, n_params_non_embedding=n_params_ne,
        final_train=history[-1]["train"], final_val=history[-1]["val"],
        best_val=min(h["val"] for h in history), history=history,
        seconds=time.time() - t0, device=device,
    )
    os.makedirs(cfg["out_dir"], exist_ok=True)
    with open(os.path.join(cfg["out_dir"], "result.json"), "w") as f:
        json.dump(result, f, indent=2)
    if cfg["save_ckpt"]:
        torch.save(
            dict(model=model.state_dict(), model_config=mc.to_dict(), chars=ds.chars),
            os.path.join(cfg["out_dir"], "ckpt.pt"),
        )
    return result


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    for k, v in DEFAULTS.items():
        if isinstance(v, bool):
            p.add_argument(f"--{k}", type=lambda s: s.lower() in ("1", "true", "yes"), default=v)
        else:
            p.add_argument(f"--{k}", type=type(v), default=v)
    args = vars(p.parse_args())
    train(**args)

In [ ]:
%%writefile sweep.py
"""Scaling study: train several model sizes x seeds (plus an optional LayerNorm-placement ablation).

Resumable: finished runs (result.json exists) are skipped, so you can re-run after a Colab disconnect.
  python sweep.py --out results                # full sweep (about 12 runs, plus ablation with --ablate)
  python sweep.py --out results --quick        # tiny CPU smoke test
"""
import argparse
import os

from train import train

# name: (n_layer, n_head, n_embd)
SIZES = {"xs": (2, 2, 64), "s": (4, 4, 128), "m": (6, 6, 192), "l": (6, 6, 384)}

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--out", default="results")
    p.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    p.add_argument("--sizes", nargs="+", default=list(SIZES))
    p.add_argument("--max_iters", type=int, default=3000)
    p.add_argument("--ablate", action="store_true", help="also run post-LN variants for each size")
    p.add_argument("--quick", action="store_true", help="tiny smoke test for CPU")
    a = p.parse_args()

    extra = {}
    if a.quick:
        a.sizes, a.seeds, a.max_iters = ["xs", "s"], [0, 1], 60
        extra = dict(batch_size=16, block_size=32, eval_interval=30, eval_iters=5)

    placements = ["pre", "post"] if a.ablate else ["pre"]
    for ln in placements:
        for size in a.sizes:
            n_layer, n_head, n_embd = SIZES[size]
            for seed in a.seeds:
                name = f"{size}_{ln}_seed{seed}"
                out_dir = os.path.join(a.out, name)
                if os.path.exists(os.path.join(out_dir, "result.json")):
                    print(f"skip {name} (done)")
                    continue
                print(f"\n=== {name} ===", flush=True)
                train(
                    n_layer=n_layer, n_head=n_head, n_embd=n_embd, ln_placement=ln, seed=seed,
                    max_iters=a.max_iters, out_dir=out_dir, save_ckpt=False, **extra,
                )

In [ ]:
%%writefile plot.py
"""Aggregate sweep results into plots and a markdown table.
  python plot.py --results results --out figures
"""
import argparse
import glob
import json
import os
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


# two-sided 95% Student-t critical values by degrees of freedom (df = n - 1)
T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228,
       11: 2.201, 12: 2.179, 13: 2.160, 14: 2.145, 15: 2.131, 16: 2.120, 17: 2.110, 18: 2.101, 19: 2.093, 20: 2.086,
       25: 2.060, 30: 2.042}


def t_crit(df):
    """Student-t multiplier for a 95% interval; nearest tabulated df at or below `df`, 1.96 beyond 30."""
    if df < 1:
        return 0.0
    if df > 30:
        return 1.96
    return T95[max(k for k in T95 if k <= df)]


def load(results_dir):
    runs = []
    for path in sorted(glob.glob(os.path.join(results_dir, "*", "result.json"))):
        with open(path) as f:
            r = json.load(f)
        name = os.path.basename(os.path.dirname(path))
        size, ln, _ = name.split("_")
        r.update(size=size, ln=ln, name=name)
        runs.append(r)
    return runs


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--results", default="results")
    p.add_argument("--out", default="figures")
    p.add_argument("--metric", choices=["best_val", "final_val"], default="best_val",
                   help="best_val = lowest validation loss over eval checkpoints (early-stopping proxy); final_val = last checkpoint")
    a = p.parse_args()
    os.makedirs(a.out, exist_ok=True)

    runs = load(a.results)
    if not runs:
        raise SystemExit(f"No result.json files found in {a.results}")

    groups = defaultdict(list)
    for r in runs:
        groups[(r["ln"], r["size"])].append(r)
    stats = {}
    for key, rs in groups.items():
        vals = np.array([r[a.metric] for r in rs])
        stats[key] = dict(
            n=len(rs), params=rs[0]["n_params"], params_ne=rs[0]["n_params_non_embedding"],
            mean=vals.mean(), std=vals.std(ddof=1) if len(vals) > 1 else 0.0,
            ci95=t_crit(len(vals) - 1) * vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0,
        )

    # 1) scaling plot (pre-LN)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    slope_txt = ""
    for ln, marker in (("pre", "o"), ("post", "s")):
        ks = sorted([k for k in stats if k[0] == ln], key=lambda k: stats[k]["params"])
        if not ks:
            continue
        x = np.array([stats[k]["params"] for k in ks], dtype=float)
        y = np.array([stats[k]["mean"] for k in ks])
        e = np.array([stats[k]["std"] for k in ks])
        ax.errorbar(x, y, yerr=e, marker=marker, capsize=3, label=f"{ln}-LN (mean +/- std over seeds)")
        if ln == "pre" and len(x) >= 2:
            slope, intercept = np.polyfit(np.log10(x), np.log10(y), 1)
            xs = np.linspace(x.min(), x.max(), 50)
            ax.plot(xs, 10 ** (intercept + slope * np.log10(xs)), "--", alpha=0.6, label=f"log-log fit, slope {slope:.3f}")
            slope_txt = f"Log-log slope of validation loss vs. parameters (pre-LN): {slope:.3f}\n"
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Parameters (total)"); ax.set_ylabel(f"Validation loss, {a.metric} (nats/char)")
    ax.set_title("Scaling: validation loss vs. model size"); ax.legend(); ax.grid(alpha=0.3, which="both")
    fig.tight_layout(); fig.savefig(os.path.join(a.out, "scaling.png"), dpi=150); plt.close(fig)

    # 2) training curves (seed 0, pre-LN)
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for r in runs:
        if r["ln"] == "pre" and r["name"].endswith("seed0"):
            ax.plot([h["iter"] for h in r["history"]], [h["val"] for h in r["history"]], label=f"{r['size']} ({r['n_params']/1e6:.2f}M)")
    ax.set_xlabel("Iteration"); ax.set_ylabel("Validation loss"); ax.set_title("Validation loss during training (seed 0)")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(os.path.join(a.out, "curves.png"), dpi=150); plt.close(fig)

    # 3) table
    lines = ["| LN | size | params | non-embedding params | n seeds | val loss (mean, " + a.metric + ") | std | 95% CI half-width (Student-t) |",
             "|---|---|---|---|---|---|---|---|"]
    for k in sorted(stats, key=lambda k: (k[0], stats[k]["params"])):
        s = stats[k]
        lines.append(f"| {k[0]} | {k[1]} | {s['params']:,} | {s['params_ne']:,} | {s['n']} | {s['mean']:.4f} | {s['std']:.4f} | {s['ci95']:.4f} |")
    md = "\n".join(lines) + "\n\n" + slope_txt
    with open(os.path.join(a.out, "summary.md"), "w") as f:
        f.write(md)
    print(md)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sample.py
"""Generate text from a saved checkpoint: python sample.py --ckpt out/run/ckpt.pt --prompt "ROMEO:" """
import argparse

import torch

from model import GPT, GPTConfig

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", default="out/run/ckpt.pt")
    p.add_argument("--prompt", default="\n")
    p.add_argument("--tokens", type=int, default=400)
    p.add_argument("--temperature", type=float, default=0.8)
    p.add_argument("--top_k", type=int, default=40)
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(a.seed)
    ck = torch.load(a.ckpt, map_location=device)
    model = GPT(GPTConfig(**ck["model_config"])).to(device)
    model.load_state_dict(ck["model"])
    model.eval()
    stoi = {c: i for i, c in enumerate(ck["chars"])}
    itos = {i: c for c, i in stoi.items()}
    idx = torch.tensor([[stoi[c] for c in a.prompt if c in stoi]], dtype=torch.long, device=device)
    out = model.generate(idx, a.tokens, temperature=a.temperature, top_k=a.top_k)
    print("".join(itos[int(i)] for i in out[0]))

In [ ]:
%%writefile tests/test_model.py
"""Sanity tests. Run:  python tests/test_model.py   (or: pytest -q)"""
import math
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
import torch

from model import GPT, GPTConfig


def make(ln="pre", **kw):
    cfg = GPTConfig(block_size=16, vocab_size=65, n_layer=2, n_head=2, n_embd=32, dropout=0.0, ln_placement=ln, **kw)
    return GPT(cfg)


def test_shapes():
    for ln in ("pre", "post"):
        m = make(ln)
        x = torch.randint(0, 65, (3, 16))
        logits, loss = m(x, x)
        assert logits.shape == (3, 16, 65)
        assert loss.ndim == 0


def test_causality():
    """Changing the LAST token must not change logits at earlier positions."""
    torch.manual_seed(0)
    m = make().eval()
    x = torch.randint(0, 65, (2, 16))
    x2 = x.clone()
    x2[:, -1] = (x2[:, -1] + 1) % 65
    l1, _ = m(x)
    l2, _ = m(x2)
    assert torch.allclose(l1[:, :-1], l2[:, :-1], atol=1e-5)
    assert not torch.allclose(l1[:, -1], l2[:, -1], atol=1e-5)


def test_initial_loss_is_near_uniform():
    torch.manual_seed(0)
    m = make().eval()
    x = torch.randint(0, 65, (8, 16))
    _, loss = m(x, x)
    assert abs(loss.item() - math.log(65)) < 0.3, loss.item()


def test_can_overfit_one_batch():
    torch.manual_seed(0)
    m = make()
    x = torch.randint(0, 65, (4, 16))
    y = torch.roll(x, -1, dims=1)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    _, first = m(x, y)
    for _ in range(150):
        _, loss = m(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
    assert loss.item() < 0.5 * first.item(), (first.item(), loss.item())


def test_generate_length():
    m = make().eval()
    out = m.generate(torch.zeros((1, 1), dtype=torch.long), 20, top_k=10)
    assert out.shape == (1, 21)


if __name__ == "__main__":
    for name, fn in list(globals().items()):
        if name.startswith("test_"):
            fn()
            print("PASS", name)

## 2. Sanity tests (run these first)
All five must print PASS. **If any fails, do not train: debug first.**

In [ ]:
!python tests/test_model.py

## 3. Quick training run (a few minutes on a T4)
Watch the validation loss fall. It should start near 4.17 (= ln 65) and drop well below 2.

In [ ]:
from train import train
r = train(max_iters=1000, eval_interval=100, out_dir='out/quick')
print('final val loss:', round(r['final_val'], 4), '| params:', r['n_params'])

In [ ]:
!python sample.py --ckpt out/quick/ckpt.pt --prompt "ROMEO:" --tokens 300

**Question for you:** is the generated text good? What kinds of mistakes does it make, and why would a larger model or longer training help?

## 4. Scaling sweep
4 sizes x 3 seeds (about 12 runs). Add `--ablate` to also run post-LN variants (doubles the time). Time depends on the GPU Colab gives you.

In [ ]:
!python sweep.py --out $RESULTS_DIR

In [ ]:
# Optional ablation (post-LN). Uncomment to run:
# !python sweep.py --out $RESULTS_DIR --ablate

## 5. Plots and summary table
`--metric best_val` (default) uses the lowest validation loss over the evaluation checkpoints, which guards against overfitting in the larger models. Re-run with `--metric final_val` to compare.

In [ ]:
!python plot.py --results $RESULTS_DIR --out figures

In [ ]:
from IPython.display import Image, display
display(Image('figures/scaling.png'))
display(Image('figures/curves.png'))

In [ ]:
print(open('figures/summary.md').read())

## 6. Download your results
Zip the figures and results so you can add them to your GitHub repo.

In [ ]:
!zip -r my_results.zip figures $RESULTS_DIR -x '*.pt'
from google.colab import files
files.download('my_results.zip')

## What to write up
Use the template in README.md: question, setup, results table with confidence intervals, what surprised you, what you got wrong, how to reproduce.